# 01 — Exploratory Data Analysis
Pertanyaan: bagaimana distribusi kerusakan, kualitas data, umur, dan kaitan material dengan grade? Metrik akhir adalah F1-macro.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd() if (Path.cwd() / 'scripts').is_dir() else Path.cwd().parent
assert (ROOT / 'scripts').is_dir(), 'Buka notebook dari root proyek atau notebooks/'
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))


In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid')
train = pd.read_csv(ROOT / 'data/train.csv')
test = pd.read_csv(ROOT / 'data/test.csv')
metadata_path = ROOT / 'reports/metadata.csv'
metadata = pd.read_csv(metadata_path) if metadata_path.exists() else pd.DataFrame()
FIGURES = ROOT / 'reports/figures'
FIGURES.mkdir(parents=True, exist_ok=True)
summary = {'train_shape': list(train.shape), 'test_shape': list(test.shape),
           'missing_train': train.isna().sum().to_dict(),
           'target_counts': train.damage_grade.value_counts().sort_index().to_dict(),
           'age_sentinel_count': int((train.age_building >= 995).sum()),
           'overlapping_building_ids': len(set(train.building_id) & set(test.building_id))}
(ROOT / 'reports/eda_summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
print(json.dumps(summary, indent=2))


## Distribusi target
Periksa ketimpangan kelas. F1-macro memberi bobot yang sama pada setiap kelas, sehingga performa kelas minoritas perlu diperiksa terpisah.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
train.damage_grade.value_counts().sort_index().plot.bar(ax=ax, color='#246a8d')
ax.set(title='Distribusi tingkat kerusakan', xlabel='Damage grade', ylabel='Jumlah bangunan')
ax.tick_params(axis='x', rotation=0)
fig.tight_layout()
fig.savefig(FIGURES / 'target_distribution.png', dpi=150)
fig


## Umur bangunan
Nilai >=995 diperlakukan sebagai sentinel, bukan umur riil. Transformasi menjadi missing dilakukan di pipeline dan median dipelajari hanya dari training fold.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
train.loc[train.age_building < 995, 'age_building'].plot.hist(bins=50, ax=ax, color='#388e7b')
ax.set(title='Distribusi umur bangunan (sentinel dikeluarkan)', xlabel='Umur (tahun)', ylabel='Jumlah bangunan')
fig.tight_layout()
fig.savefig(FIGURES / 'age_distribution.png', dpi=150)
fig


## Material dan target
Bandingkan proporsi, bukan hanya jumlah absolut. Asosiasi material dan kerusakan juga dapat dipengaruhi wilayah serta faktor lain; grafik tidak membuktikan hubungan kausal.

In [ ]:
proportions = pd.crosstab(train.superstructure, train.damage_grade, normalize='index')
proportions.index = proportions.index.str.replace('has_superstructure_', '', regex=False)
fig, ax = plt.subplots(figsize=(11, 6))
proportions.plot.barh(stacked=True, ax=ax, colormap='viridis')
ax.set(title='Proporsi damage grade menurut material', xlabel='Proporsi', ylabel='Material')
ax.legend(title='Grade', bbox_to_anchor=(1.02, 1), loc='upper left')
fig.tight_layout()
fig.savefig(FIGURES / 'material_damage.png', dpi=150)
fig


## Kesimpulan
Grade 5 mendominasi training; tidak ada missing pada kolom asli, tetapi sentinel umur tetap memerlukan penanganan. Evaluasi selanjutnya memakai split stratified, preprocessing fold-local, class weighting, dan F1-macro. Fitur ID bersifat opsional dan harus diuji dengan hati-hati.